# Кто покупает кофе в «Зерне» осенью 2026 года

In [1]:
from pathlib import Path
import sys
import pandas as pd
RANDOM_STATE=42
DATA=Path("data")
FORM_FILE="clients_raw.csv"
print(sys.version.split()[0],pd.__version__)

3.12.14 3.0.6


## 1. Цель

Маркетингу нужен портрет известных гостей. Сравним частоту покупок и выручку по источникам и возрасту, отдельно посчитаем долю анкет с неизвестным возрастом и долю заказов в городе гостя.

**Вывод.** Для средних по источникам включим гостей без заказов. Для возрастных трат используем активных с известным возрастом и покажем размер каждой группы.

## 2. Данные

In [2]:
raw=pd.read_csv(DATA/FORM_FILE)
orders=pd.read_csv(DATA/"orders_3m.csv",parse_dates=["datetime"])
shops=pd.read_csv(DATA/"shops.csv")
orders["client_id"]=orders.client_id.str.strip().str.upper()
print(raw.shape,orders.shape,shops.shape)
pd.DataFrame({"тип":raw.dtypes.astype(str),"пропуски":raw.isna().sum(),"разных":raw.nunique()})

(307, 9) (2089, 9) (7, 5)


,тип,пропуски,разных
client_id,str,0,299
name,str,0,33
gender,str,19,6
birth_date,str,19,249
city,str,16,29
phone,str,17,277
signup_date,str,0,266
bonus,str,15,138
source,str,27,14


In [3]:
print(raw.city.value_counts(dropna=False))
print(raw.duplicated().sum(),raw.client_id.nunique())

city
Пермь              76
Казань             73
Тюмень             30
Уфа                19
NaN                16
Екатеринбург       11
 Тюмень             7
пермь               7
г. Пермь            7
Москва              7
г. Казань           6
г. Тюмень           6
Казань              5
казань              5
 Казань             5
ЕКБ                 5
Пермь               4
Екб                 3
Тюмень              2
 Пермь              2
г. Уфа              2
Москва              1
екатеринбург        1
г.Пермь             1
 Уфа                1
г. Екатеринбург     1
Уфа                 1
тюмень              1
Екатеринбург        1
уфа                 1
Name: count, dtype: int64
8 299


**Вывод.** В выгрузке 307 строк. Среди городов встречаются разные варианты регистра и префиксов. Встречаются полные копии строк и несколько анкет одной карты.

## 3. Предобработка

In [4]:
SYNONYMS = {"соц. сети": "соцсети", "от друзей": "друзья", "проходил мимо": "мимо",
            "2гис": "карты", "яндекс карты": "карты"}

def clean_city(s):
    return (s.str.strip().str.lower().str.removeprefix("гор.").str.removeprefix("г.").str.removesuffix(", россия").str.strip()
             .replace({"екб": "екатеринбург"}).str.capitalize())

def clean_phone(s):
    digits = s.str.replace(r"\D", "", regex=True)
    ok = digits.str.fullmatch(r"[78]?\d{10}")      # 10 цифр или 11 с кодом 7/8
    return digits.where(ok).str[-10:]               # другие длины — пропуск, а не «похожий» номер

def clean_gender(s):
    return s.str.strip().str.lower().map({"м":"М","муж":"М","ж":"Ж","жен":"Ж"})

def parse_birth(s):
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    ru = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
    born = iso.fillna(ru)
    return born.where(born.between("1926-12-01", "2012-12-01"))

def clean_bonus(s):
    b = pd.to_numeric(s.str.replace(" ", ""), errors="coerce")
    return b.mask((b < 0) | (b > 10_000))

def clean_clients(raw):
    d=raw.drop_duplicates().copy()
    d["client_id"]=d.client_id.str.strip().str.upper()
    d["name"]=d.name.str.strip().str.capitalize()
    for col,fn in [("gender",clean_gender),("birth_date",parse_birth),("city",clean_city),("phone",clean_phone),("bonus",clean_bonus)]:
        d[col]=fn(d[col])
    d["signup_date"]=pd.to_datetime(d.signup_date)
    d["source"]=d.source.str.strip().str.lower().replace(SYNONYMS)
    return d.sort_values("signup_date").drop_duplicates("client_id",keep="last",ignore_index=True)

In [5]:
def check_clients(df):
    assert df.client_id.notna().all() and df.client_id.is_unique, "Проверьте ключ карты"
    assert df.gender.dropna().isin(["М","Ж"]).all(), "Проверьте пол"
    assert df.city.dropna().isin(["Пермь","Казань","Тюмень","Уфа","Екатеринбург","Москва"]).all(), "Проверьте город"
    assert df.bonus.dropna().between(0,10000).all(), "Проверьте бонус"


clients=clean_clients(raw)
check_clients(clients)
clients.to_csv(DATA/"clients_clean.csv",index=False)
log={"выгрузка":len(raw),"без полных дублей":len(raw.drop_duplicates()),"анкет":len(clients)}
log

{'выгрузка': 307, 'без полных дублей': 299, 'анкет': 293}

**Вывод.** Из 307 строк убрали 8 точных копий и 6 повторных анкет. Сохранилась свежая по дате анкета каждого из 293 гостей.

## 3.1. Качество данных

In [6]:
missing_pct=(clients.isna().mean()*100).round(1).sort_values(ascending=False)
usable_age=round(clients.birth_date.notna().mean()*100,1)
missing_pct

birth_date     14.3
source          9.2
bonus           7.8
gender          6.1
city            5.5
phone           5.5
client_id       0.0
name            0.0
signup_date     0.0
dtype: float64

In [7]:
b=pd.to_numeric(raw.bonus.str.replace(" ",""),errors="coerce")
iso=pd.to_datetime(raw.birth_date,format="%Y-%m-%d",errors="coerce")
ru=pd.to_datetime(raw.birth_date,format="%d.%m.%Y",errors="coerce")
born=iso.fillna(ru)
def ticket(col,rule,mask,condition):
    return {"где":FORM_FILE+" / "+col,"правило":rule,"строк":int(mask.sum()),"всего":len(raw),"доля_%":round(mask.mean()*100,1),"примеры":raw.loc[mask,"client_id"].head().tolist(),"как_найти":condition}
tickets=[ticket("bonus","Баланс между 0 и 10000",b.lt(0)|b.gt(10000),"преобразовать в число и проверить границы"),ticket("birth_date","1900-01-01 — служебное значение",born.eq("1900-01-01"),"разобрать ISO и дд.мм.гггг, отобрать 1900-01-01"),ticket("birth_date","Дата не должна быть в будущем",born.gt("2026-12-01"),"дата > 2026-12-01")]
pd.DataFrame(tickets)

,где,правило,строк,всего,доля_%,примеры,как_найти
0,clients_raw.csv / bonus,Баланс между 0 и 10000,3,307,1.0,"[C1095, C1188, C1033]",преобразовать в число и проверить границы
1,clients_raw.csv / birth_date,1900-01-01 — служебное значение,3,307,1.0,"[C1140, C1061, C1007]","разобрать ISO и дд.мм.гггг, отобрать 1900-01-01"
2,clients_raw.csv / birth_date,Дата не должна быть в будущем,1,307,0.3,[C1203],дата > 2026-12-01


**Вывод.** Возраст известен в 85.7 % анкет. Владелец CRM может воспроизвести три проверки по маскам. Неизвестный возраст оставили пропуском.

## 4.1. Возраст и города

In [8]:
b=clients.birth_date
clients["age"]=2026-b.dt.year-((b.dt.month==12)&(b.dt.day>1)).astype(int)
clients["age_group"]=pd.cut(clients.age,[14,25,35,45,60,100],right=False,labels=["14–24","25–34","35–44","45–59","60–99"])
median_age=clients.age.median()
age_counts=clients.age_group.value_counts().to_dict()
by_city=clients.city.value_counts().to_dict()
print(median_age,age_counts,by_city)

39.0 {'45–59': 84, '25–34': 61, '35–44': 60, '14–24': 38, '60–99': 8} {'Пермь': 89, 'Казань': 88, 'Тюмень': 46, 'Уфа': 24, 'Екатеринбург': 22, 'Москва': 8}


**Вывод.** Медиана — 39 лет среди 251 анкет с датой рождения. Самая большая возрастная группа — 45–59, в ней 84 человек.

## 4.2. Кто сделал заказ

In [9]:
stats=orders.groupby("client_id").agg(orders=("order_id","count"),revenue=("amount","sum"))
guests=clients.merge(stats,on="client_id",how="left",validate="one_to_one").fillna({"orders":0,"revenue":0})
active_pct=round((guests.orders>0).mean()*100,1)
n_orphans=len(set(stats.index)-set(clients.client_id))
print(active_pct,n_orphans)

88.7 0


**Вывод.** Среди 293 анкет 88.7 % совершили покупку. В заказах найдено 0 разных номеров карты без анкеты. Заказы без карты нельзя приписать конкретному гостю.

## 4.3. Источники

In [10]:
src=guests.source.fillna("не указано")
vc=src.value_counts()
groups=src.where(src.isin(vc[vc>=10].index),"прочее")
d=guests.assign(active=guests.orders>0)
by_source=d.groupby(groups).agg(clients=("client_id","size"),active_pct=("active","mean"),avg_revenue=("revenue","mean")).sort_values("clients",ascending=False)
by_source.active_pct=(by_source.active_pct*100).round(1)
by_source.avg_revenue=by_source.avg_revenue.round()
best_source=by_source.loc[by_source.clients>=20,"avg_revenue"].idxmax()
by_source

,clients,active_pct,avg_revenue
source,,,
соцсети,80,87.5,1545.0
друзья,78,89.7,1431.0
мимо,41,87.8,1327.0
карты,32,87.5,1481.0
не указано,27,85.2,1523.0
реклама,27,92.6,1449.0
прочее,8,100.0,1340.0


**Вывод.** Среди источников с 20 и более анкетами лидирует «соцсети» — 1545 ₽ на гостя за осень, 80 анкет. Это сравнение описывает зарегистрировавшихся гостей.

## 4.4. Выручка по возрасту

In [11]:
by_age=guests[guests.orders>0].groupby("age_group",observed=True).agg(guests_n=("client_id","size"),avg_orders=("orders","mean"),avg_revenue=("revenue","mean")).round(1)
best_group=by_age.avg_revenue.idxmax()
by_age

,guests_n,avg_orders,avg_revenue
age_group,,,
14–24,35,5.2,1744.6
25–34,54,4.5,1477.0
35–44,57,4.6,1701.3
45–59,70,4.9,1655.7
60–99,8,5.0,1681.9


**Вывод.** В группе 14–24 самая высокая средняя выручка среди активных гостей с известным возрастом — 1744.6 ₽, 35 гостей. Размеры групп указаны в таблице.

## 4.5. Город покупки

In [12]:
d=orders.dropna(subset=["client_id"]).merge(shops[["shop_id","city"]],on="shop_id",how="left").merge(clients[["client_id","city"]],on="client_id",how="left",suffixes=("_shop","_home"))
known=d.dropna(subset=["city_shop","city_home"])
home_pct=round((known.city_shop==known.city_home).mean()*100,1)
travelers=(d.groupby("client_id").city_shop.nunique()>=2).sum()
print(home_pct,travelers,len(known))

85.1 73 1183


**Вывод.** Для 1183 заказов известны оба города, 85.1 % пришлись на город гостя. В нескольких городах заказывали 73 гостей.

## 5. Выводы и рекомендации

In [13]:
def report(raw, orders):
    clients=clean_clients(raw)
    b=clients.birth_date
    age=2026-b.dt.year-((b.dt.month==12)&(b.dt.day>1)).astype(int)
    rev=orders.groupby("client_id").amount.sum()
    amounts=clients.client_id.map(rev)
    return dict(clients=len(clients),active_pct=round(amounts.notna().mean()*100,1),median_age=age.median(),avg_revenue_active=round(amounts.mean(),1))


result=report(raw,orders)
result

{'clients': 293,
 'active_pct': np.float64(88.7),
 'median_age': np.float64(39.0),
 'avg_revenue_active': np.float64(1646.0)}

**Что сделали.** Связали 293 очищенные анкеты с покупками за осень.

**Что нашли.** У активного гостя в среднем 1646.0 ₽ покупок. Возраст известен в 85.7 % анкет.

**Что предлагаем.** Перед расширением привлечения проверить на сопоставимых группах канал «соцсети». Владельцу CRM исправить появление служебных дат и недопустимых балансов.